In [2]:
import pandas as pd
import numpy as np


# Load the data
btc = pd.read_csv("BTC_full_data.csv")
eth = pd.read_csv("ETH_full_data.csv")

# Clean data (remove first row due to missing Log_Return value and convert cols to numeric)
def clean_data(data):
    df = data
    # remove first row
    df = df.iloc[1:].reset_index(drop=True)

    df["Date"] = pd.to_datetime(df["Date"])
    # convert numeric columns
    for col in df.columns[1:]:
        df[col] = pd.to_numeric(df[col], errors="coerce")
    df = df.set_index("Date")

    return df

btc = clean_data(btc)
eth = clean_data(eth)

In [3]:
def breakout_signal(df):

    data = df.copy()

    # 20 day breakout levels
    data['high20'] = data['Close'].shift(1).rolling(20).max()
    data['low20'] = data['Close'].shift(1).rolling(20).min()

    # signal
    data['signal'] = 0
    data.loc[data['Close'] > data['high20'], 'signal'] = 1
    data.loc[data['Close'] < data['low20'], 'signal'] = -1

    # hold previous position
    data['position'] = data['signal'].replace(0, np.nan).ffill().fillna(0)

    return data

In [4]:
btc = breakout_signal(btc)
eth = breakout_signal(eth)

In [13]:

def build_trade_table(df):
    data = breakout_signal(df)

    # 2. Trade column (your signal)
    data['trade'] = data['signal']

    # 3. Trade action (readable labels)
    data['trade_action'] = 'hold'
    data.loc[data['trade'] == 1, 'trade_action'] = 'buy'
    data.loc[data['trade'] == -1, 'trade_action'] = 'sell'

    # 4. Rename columns
    data = data.rename(columns={
        'Close': 'price'
    })

    # 5. Select final table
    final_table = data[['price', 'Log_Return', 'trade', 'trade_action', 'position']]

    # 6. If your index is date → reset it
    final_table = final_table.reset_index().rename(columns={'index': 'date'})

    return final_table

In [21]:
btc_table = build_trade_table(btc)
eth_table = build_trade_table(eth)



In [23]:
btc_table.to_csv('btc_turtle_table.csv', index=True)
eth_table.to_csv('eth_turtle_table.csv', index=True)